# YOLO26n: 2D chess piece and board detection
Generate synthetic screenshots, verify labels, train a pretrained YOLO26n, and evaluate/export the best checkpoint.

The shared recipe in `yolo26_chess.yaml` is a research-informed baseline, not a measured optimum. Select parameters on validation data and reserve real screenshots for final FEN evaluation.


In [ ]:
# 1. Install Ultralytics and dependencies
!pip install -q "ultralytics>=8.4.0,<9"

import os
import time
import torch
import ultralytics

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
print(f"Ultralytics Version: {ultralytics.__version__}")

In [ ]:
# 2. Clone or update the project repository
REPO_DIR = "/kaggle/working/2d-chess-pieces-detection"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Zai-Kun/2d-chess-pieces-detection {REPO_DIR}
else:
    print(f"Repository already present at: {REPO_DIR}")

%cd {REPO_DIR}
%pip install -q -r requirements.txt


## Dataset generation and QA
Full-board, app UI, embedded, and multi-board scenes, with tight alpha boxes and moderate screenshot artifacts.
All rendering styles appear in training; background files are disjoint and sample seeds are split-specific. Use `--holdout-styles` for an unseen-style benchmark instead.
Twenty percent of positions explicitly show all 12 piece classes at varied squares. The 18-pixel floor is enforced, including the final retry.


In [ ]:
# 3. Generate high-quality dataset
# Start with 20,000 train and 1,500 images per evaluation split; reduce counts for a smoke test.
import os
import subprocess

cmd = [
    "python3", "generate_datasets.py",
    "--train-count", "20000",
    "--val-count", "1500",
    "--test-count", "1500",
    "--imgsz", "800",
    "--seed", "42",
    "--clean"
]
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise RuntimeError(f"Dataset generation failed with return code {ret.returncode}! Stopping pipeline.")

if not (os.path.exists("datasets/images/train") and len(os.listdir("datasets/images/train")) > 0):
    raise RuntimeError("Training images not found after generation! Stopping pipeline.")


In [ ]:
# 4. Run Automated QA Suite
# Checks 1:1 pair integrity, coordinate ranges, boundary compliance, and duplicate prevention
import subprocess

ret = subprocess.run(["python3", "verify_dataset.py", "--data-dir", "datasets"])
if ret.returncode != 0:
    raise RuntimeError("Dataset verification failed QA checks! Stopping pipeline.")


In [ ]:
# 5. Display sample preview montage with bounding boxes
from PIL import Image
import matplotlib.pyplot as plt

preview_path = "results/previews/preview_train.jpg"
if os.path.exists(preview_path):
    plt.figure(figsize=(15, 10))
    plt.imshow(Image.open(preview_path))
    plt.axis("off")
    plt.title("Sample Generated Training Images with YOLO Bounding Boxes", fontsize=14)
    plt.show()

## Training
Use the same settings as the CLI: 800 pixels, 100 epochs, patience 30, MuSGD, and conservative screenshot augmentation. Batch size depends on available GPU memory.
For hyperparameter search, use `train_yolo26.py --tune 20 --epochs 60` separately, then retrain with its `--hyp` file. Compare full runs before selecting a winner; short searches can favor fast convergence.


In [ ]:
# Train with the shared recipe and a Kaggle session time limit.
import torch
from train_yolo26 import main as train_yolo26

device = "0" if torch.cuda.is_available() else "cpu"
batch = 16
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device, batch = "0,1", 32
model = train_yolo26(["--device", device, "--batch", str(batch), "--time", "10.5"])


## Final evaluation
Inspect validation metrics while developing. Run the test evaluation only after choosing the final recipe, and keep real screenshot tests separate from tuning.


In [ ]:
# Evaluate exactly the checkpoint produced above.
from pathlib import Path
from ultralytics import YOLO

run_dir = str(model.trainer.save_dir)
best_weights = str(model.trainer.best)
assert Path(best_weights).is_file(), "Training did not produce best.pt"
data_yaml = str(Path("datasets/data.yaml").resolve())
train_imgsz = model.trainer.args.imgsz
best_model = YOLO(best_weights)

val_metrics = best_model.val(data=data_yaml, split="val", imgsz=train_imgsz)
print(f"Val mAP50: {val_metrics.box.map50:.4f}; mAP50-95: {val_metrics.box.map:.4f}")
test_metrics = best_model.val(data=data_yaml, split="test", imgsz=train_imgsz)
print(f"Test mAP50: {test_metrics.box.map50:.4f}; mAP50-95: {test_metrics.box.map:.4f}")


In [ ]:
# 8. Display training curves and confusion matrix
if not run_dir or not os.path.exists(run_dir):
    run_dir = os.path.dirname(os.path.dirname(best_weights))
for plot_file in ["results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    full_path = os.path.join(run_dir, plot_file)
    if os.path.exists(full_path):
        plt.figure(figsize=(10, 6))
        plt.imshow(Image.open(full_path))
        plt.axis("off")
        plt.title(plot_file, fontsize=13)
        plt.show()

## 5. Visualizing Detections on Novel Test Images

In [ ]:
# 9. Run inference on random test samples and visualize predictions
import glob
import random

test_images = sorted(glob.glob("datasets/images/test/*.jpg"))
if test_images:
    selected = random.sample(test_images, min(4, len(test_images)))
    preds = best_model.predict(source=selected, imgsz=train_imgsz, conf=0.25)

    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    for idx, res in enumerate(preds):
        ax = axes[idx // 2, idx % 2]
        ax.imshow(res.plot()[:, :, ::-1])  # BGR to RGB
        ax.axis("off")
        ax.set_title(f"Test Prediction {idx + 1} ({len(res.boxes)} objects)", fontsize=12)
    plt.tight_layout()
    plt.show()

## ONNX export
Export the NMS-free head explicitly and validate the exported artifact before deployment. Compare its accuracy and latency against the default inference path for your application.


In [ ]:
# 10. Export to ONNX and copy artifacts to /kaggle/working root
import shutil

# Export to ONNX (with native end-to-end NMS-free inference)
onnx_path = best_model.export(format="onnx", imgsz=train_imgsz, dynamic=False, nms=False)
print(f"Exported ONNX model: {onnx_path}")

output_root = "/kaggle/working"
artifacts = [
    (best_weights, os.path.join(output_root, "yolo26n_chess_best.pt")),
    (onnx_path, os.path.join(output_root, "yolo26n_chess_best.onnx")),
    (os.path.join(run_dir, "results.png"), os.path.join(output_root, "training_results.png")),
    (os.path.join(run_dir, "confusion_matrix.png"), os.path.join(output_root, "confusion_matrix.png")),
]

print("\nPackaging outputs for direct Kaggle download:")
for src, dst in artifacts:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  -> Copied {os.path.basename(dst)} to {output_root}")

print("\n🎉 Done! All artifacts are ready in Kaggle's Output panel for download.")